# Session 3: RAG - Retrieval Augmented Generation (55 minutes)

## 🎯 Learning Objectives
- Understand the RAG pattern and why it's essential
- Load and process documents
- Create vector embeddings
- Build a document Q&A system

## 📋 Problem Statement
Our Research Assistant needs to answer questions about **YOUR** documents!
- PDFs, text files, websites
- Company knowledge bases
- Private documentation

## ⏱️ Session Breakdown
- 10 min: The RAG concept
- 15 min: Document loading and chunking
- 15 min: Embeddings and vector stores
- 10 min: Building the Q&A chain
- 5 min: Recap

---

## 🆓 Using Qwen 0.5B via Ollama (LOCAL, FREE!)

## 1. Setup & Dependencies

In [ ]:
# Install RAG dependencies if needed
# !pip install chromadb sentence-transformers

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma

# Initialize Qwen 0.5B via Ollama (LOCAL, FREE!)
llm = ChatOllama(
    model="qwen2:0.5b",
    temperature=0.7
)

print("✅ Session 3 Setup Complete!")
print("🖥️  Using qwen2:0.5b via Ollama - LOCAL, FREE, NO LIMITS!")

✅ Session 3 Setup Complete!
🖥️  Using qwen2:0.5b via Ollama - LOCAL, FREE, NO LIMITS!


## 2. The Problem: LLMs Don't Know YOUR Data!

🎯 **Problem**: LLMs have a knowledge cutoff and don't know about:
- Your company's documents
- Recent information
- Private/confidential data

❌ **Without RAG:**

In [3]:
# Ask about a fictional company policy
response = llm.invoke("What is TechCorp's vacation policy for senior engineers?")
print("❌ Without RAG (LLM doesn't know your company):")
print(response.content)

❌ Without RAG (LLM doesn't know your company):
As an AI language model, I don't have access to the most up-to-date information on specific companies' policies regarding vacation policies. However, in general, many large corporations offer flexible scheduling and options for remote work or part-time schedules, which can provide some flexibility for senior engineering professionals who may want to take time off or reduce their workload.

It's important for senior engineers to carefully review the terms and conditions of their vacation policy and other company policies before deciding if they are suitable for them. Some companies may offer more flexible options than others, and it's worth reaching out to HR or a representative at TechCorp to ask about these factors.


## 3. RAG Architecture Explained

```
┌─────────────────────────────────────────────────────────────┐
│                    RAG ARCHITECTURE                          │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│   📄 DOCUMENTS                                               │
│      │                                                       │
│      ▼                                                       │
│   📦 CHUNK (Split into smaller pieces)                       │
│      │                                                       │
│      ▼                                                       │
│   🔢 EMBED (Convert to vectors)                              │
│      │                                                       │
│      ▼                                                       │
│   🗄️ STORE (Vector database)                                 │
│                                                              │
│  ─────────────────────────────────────────                   │
│                                                              │
│   ❓ QUERY                                                   │
│      │                                                       │
│      ▼                                                       │
│   🔍 RETRIEVE (Find similar chunks)                          │
│      │                                                       │
│      ▼                                                       │
│   🤖 GENERATE (LLM + Context → Answer)                       │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

## 4. Step 1: Document Loading

LangChain provides loaders for many document types:
- PDF, Word, Excel
- Web pages, Wikipedia
- Databases, APIs
- And more!

In [4]:
from langchain_core.documents import Document

# For this demo, we'll create sample documents
# In production, you'd use actual document loaders

sample_documents = [
    Document(
        page_content="""TechCorp Employee Handbook - Vacation Policy
        
        All full-time employees are entitled to paid vacation days based on their tenure:
        - 0-2 years: 15 days per year
        - 2-5 years: 20 days per year  
        - 5-10 years: 25 days per year
        - 10+ years: 30 days per year
        
        Senior Engineers (Level 4+) receive an additional 5 days annually.
        Unused vacation days can be carried over up to 10 days per year.
        Employees must give 2 weeks notice for vacation longer than 5 days.""",
        metadata={"source": "handbook.pdf", "section": "vacation"}
    ),
    Document(
        page_content="""TechCorp Employee Handbook - Remote Work Policy
        
        TechCorp supports flexible work arrangements:
        - All employees can work remotely up to 3 days per week
        - Senior staff (Level 4+) can work fully remote with manager approval
        - Core hours are 10am-3pm in your local timezone
        - Teams must have at least one in-person day per week
        
        Equipment provided for remote work:
        - Laptop, monitor, keyboard, mouse
        - $500 home office stipend annually
        - Internet reimbursement up to $75/month""",
        metadata={"source": "handbook.pdf", "section": "remote_work"}
    ),
    Document(
        page_content="""TechCorp Engineering Guidelines - Code Review Process
        
        All code changes require peer review before merging:
        
        1. Create a pull request with clear description
        2. Request review from at least 2 team members
        3. For critical systems, require Senior Engineer (L4+) approval
        4. All CI tests must pass before merge
        5. Maximum PR size: 500 lines (excluding tests)
        
        Review turnaround expectations:
        - Small PRs (<100 lines): 24 hours
        - Medium PRs (100-300 lines): 48 hours
        - Large PRs (300-500 lines): 72 hours""",
        metadata={"source": "engineering_guide.pdf", "section": "code_review"}
    ),
    Document(
        page_content="""TechCorp Engineering Guidelines - On-Call Rotation
        
        Senior Engineers participate in on-call rotation:
        
        Schedule:
        - Rotation period: 1 week
        - Primary on-call: Responds within 15 minutes
        - Secondary on-call: Backup, responds within 30 minutes
        
        Compensation:
        - $500 bonus per week of on-call duty
        - Additional compensation for incidents outside business hours
        - Comp day off after handling major incidents
        
        Escalation:
        - P0 (Critical): Wake up on-call immediately
        - P1 (High): Respond within 30 minutes
        - P2 (Medium): Respond next business day""",
        metadata={"source": "engineering_guide.pdf", "section": "oncall"}
    )
]

print(f"📄 Loaded {len(sample_documents)} documents")
for doc in sample_documents:
    print(f"  - {doc.metadata['source']} ({doc.metadata['section']})")

📄 Loaded 4 documents
  - handbook.pdf (vacation)
  - handbook.pdf (remote_work)
  - engineering_guide.pdf (code_review)
  - engineering_guide.pdf (oncall)


## 5. Using Document Loaders (optional for demo)

In [5]:
# Example: Loading from text files
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# Load a single text file (if exists)
# loader = TextLoader("data/document.txt")
# docs = loader.load()

# Load all .txt files from a directory
# loader = DirectoryLoader("data/", glob="**/*.txt")
# docs = loader.load()

# Example: Loading from PDF (requires pypdf)
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("data/document.pdf")
# docs = loader.load()

# Example: Loading from web
# from langchain_community.document_loaders import WebBaseLoader
# loader = WebBaseLoader("https://example.com")
# docs = loader.load()

print("📚 Document loaders available for many formats!")

📚 Document loaders available for many formats!


## 6. Step 2: Text Splitting (Chunking)

🎯 **Why chunk documents?**
- LLMs have token limits
- Smaller chunks = more precise retrieval
- Keep related content together

### Chunking Strategies

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # Max characters per chunk
    chunk_overlap=50,        # Overlap between chunks
    separators=["\n\n", "\n", " ", ""],  # Split priority
    length_function=len
)

# Split the documents
chunks = text_splitter.split_documents(sample_documents)

print(f"📦 Split {len(sample_documents)} documents into {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk.page_content)} chars | Source: {chunk.metadata['section']}")
    print(f"Preview: {chunk.page_content[:100]}...\n")

📦 Split 4 documents into 8 chunks

Chunk 1: 290 chars | Source: vacation
Preview: TechCorp Employee Handbook - Vacation Policy

        All full-time employees are entitled to paid v...

Chunk 2: 215 chars | Source: vacation
Preview: Senior Engineers (Level 4+) receive an additional 5 days annually.
        Unused vacation days can ...

Chunk 3: 363 chars | Source: remote_work
Preview: TechCorp Employee Handbook - Remote Work Policy

        TechCorp supports flexible work arrangement...

Chunk 4: 171 chars | Source: remote_work
Preview: Equipment provided for remote work:
        - Laptop, monitor, keyboard, mouse
        - $500 home o...

Chunk 5: 402 chars | Source: code_review
Preview: TechCorp Engineering Guidelines - Code Review Process

        All code changes require peer review ...

Chunk 6: 167 chars | Source: code_review
Preview: Review turnaround expectations:
        - Small PRs (<100 lines): 24 hours
        - Medium PRs (100...

Chunk 7: 474 chars | Source: oncall
Previ

## 7. Step 3: Embeddings

🎯 **What are embeddings?**
- Convert text to numerical vectors
- Similar meaning → Similar vectors
- Enable semantic search

We'll use the free HuggingFace embeddings!

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

# Use a free, local embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test the embeddings
test_text = "What is the vacation policy?"
test_embedding = embeddings.embed_query(test_text)

print(f"📊 Embedding dimension: {len(test_embedding)}")
print(f"Sample values: {test_embedding[:5]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📊 Embedding dimension: 384
Sample values: [0.03845001757144928, 0.06075320392847061, 0.0598783940076828, 0.01786576770246029, 0.02197304368019104]


## 8. Understanding Similarity

In [8]:
import numpy as np

def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Test semantic similarity
texts = [
    "vacation days off work",
    "PTO and holiday time",
    "code review process"
]

query = "time off from work"
query_embed = embeddings.embed_query(query)

print(f"Query: '{query}'\n")
print("Similarity scores:")
for text in texts:
    text_embed = embeddings.embed_query(text)
    similarity = cosine_similarity(query_embed, text_embed)
    print(f"  '{text}': {similarity:.4f}")

Query: 'time off from work'

Similarity scores:
  'vacation days off work': 0.6477
  'PTO and holiday time': 0.3117
  'code review process': 0.0027


## 9. Step 4: Vector Store

Store embeddings in a vector database for efficient similarity search.

We'll use Chroma (free, local, no setup needed).

In [9]:
from langchain_community.vectorstores import Chroma

# Create vector store from chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="techcorp_docs"
)

print(f"🗄️ Vector store created with {len(chunks)} chunks!")

🗄️ Vector store created with 8 chunks!


## 10. Similarity Search

In [10]:
# Test retrieval
query = "How many vacation days do senior engineers get?"

# Find similar documents
results = vectorstore.similarity_search(query, k=2)

print(f"🔍 Query: '{query}'\n")
print("📄 Retrieved Documents:")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} (Source: {doc.metadata['section']}) ---")
    print(doc.page_content[:300] + "...")

🔍 Query: 'How many vacation days do senior engineers get?'

📄 Retrieved Documents:

--- Result 1 (Source: vacation) ---
Senior Engineers (Level 4+) receive an additional 5 days annually.
        Unused vacation days can be carried over up to 10 days per year.
        Employees must give 2 weeks notice for vacation longer than 5 days....

--- Result 2 (Source: vacation) ---
TechCorp Employee Handbook - Vacation Policy

        All full-time employees are entitled to paid vacation days based on their tenure:
        - 0-2 years: 15 days per year
        - 2-5 years: 20 days per year  
        - 5-10 years: 25 days per year
        - 10+ years: 30 days per year...


## 11. Similarity Search with Scores

In [11]:
# Get results with similarity scores
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"🔍 Query: '{query}'\n")
print("📊 Results with Scores (lower = more similar):")
for doc, score in results_with_scores:
    print(f"\n  Score: {score:.4f} | Section: {doc.metadata['section']}")
    print(f"  Preview: {doc.page_content[:100]}...")

🔍 Query: 'How many vacation days do senior engineers get?'

📊 Results with Scores (lower = more similar):

  Score: 0.3181 | Section: vacation
  Preview: Senior Engineers (Level 4+) receive an additional 5 days annually.
        Unused vacation days can ...

  Score: 0.9279 | Section: vacation
  Preview: TechCorp Employee Handbook - Vacation Policy

        All full-time employees are entitled to paid v...

  Score: 1.0953 | Section: oncall
  Preview: TechCorp Engineering Guidelines - On-Call Rotation

        Senior Engineers participate in on-call ...


## 12. Creating a Retriever

Convert the vector store to a retriever for use in chains.

In [12]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 results
)

# Test the retriever
docs = retriever.invoke("remote work policy")
print(f"✅ Retriever returns {len(docs)} documents")

✅ Retriever returns 3 documents


## 13. Step 5: Building the RAG Chain

In [13]:
from langchain_core.runnables import RunnablePassthrough

# Create RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful HR assistant for TechCorp. Answer questions based ONLY on the provided context.
If the answer is not in the context, say "I don't have that information in my documents."

Context:
{context}

Question: {question}

Answer: """)

def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG Chain built successfully!")

✅ RAG Chain built successfully!


## 14. Test the RAG Chain

In [14]:
# Test with various questions
questions = [
    "How many vacation days do senior engineers get?",
    "Can I work from home?",
    "What's the code review process?",
    "How much is the on-call bonus?"
]

print("🤖 RAG Q&A SYSTEM\n" + "="*50)

for question in questions:
    print(f"\n❓ Question: {question}")
    answer = rag_chain.invoke(question)
    print(f"✅ Answer: {answer}")
    print("-" * 50)

🤖 RAG Q&A SYSTEM

❓ Question: How many vacation days do senior engineers get?
✅ Answer: 15 days per year.
--------------------------------------------------

❓ Question: Can I work from home?
✅ Answer: Yes, you can work remotely up to 3 days per week.
--------------------------------------------------

❓ Question: What's the code review process?
✅ Answer: The code review process at TechCorp Engineering Guidelines involves creating a pull request with clear descriptions, requesting peer review from at least two team members, and requiring Senior Engineer approval for critical systems. The maximum PR size is limited to 500 lines in cases of critical systems. For larger projects, the turnaround time depends on factors like the complexity of the code changes, the number of team members, and the availability of resources. It's important for teams to stay organized as they need to prepare for various stages including development, QA, CI/CD, maintenance, and release cycles.
------------------

## 15. RAG with Sources

Return the source documents alongside the answer for transparency.

In [15]:
from langchain_core.runnables import RunnableParallel

# Chain that returns both answer and sources
rag_chain_with_sources = RunnableParallel(
    {
        "answer": rag_chain,
        "sources": retriever
    }
)

# Test with sources
result = rag_chain_with_sources.invoke("What is the remote work policy?")

print("🔍 Question: What is the remote work policy?\n")
print(f"✅ Answer:\n{result['answer']}\n")
print("📚 Sources:")
for doc in result['sources']:
    print(f"  - {doc.metadata['source']} ({doc.metadata['section']})")

🔍 Question: What is the remote work policy?

✅ Answer:
All employees can work remotely up to 3 days per week, and Senior staff (Level 4+) can work fully remote with manager approval. Teams must have at least one in-person day per week. The laptop, monitor, keyboard, mouse, and $500 home office stipend annually are the equipment provided for remote work. Additionally, senior engineers participate in on-call rotation and receive a $500 bonus per week of on-call duty, along with additional compensation for incidents outside business hours. However, the compensation is limited to $500 a week.

📚 Sources:
  - handbook.pdf (remote_work)
  - handbook.pdf (remote_work)
  - engineering_guide.pdf (oncall)


## 16. Advanced: Conversational RAG

In [16]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Conversational RAG prompt
conversational_rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful HR assistant for TechCorp. 
    Answer questions based on the provided context and conversation history.
    If you don't know the answer from the context, say so."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", """Context from documents:
    {context}
    
    Question: {question}""")
])

# Simple memory for conversation
class SimpleMemory:
    def __init__(self):
        self.messages = []
    
    def save_context(self, inputs: dict, outputs: dict):
        """Save user input and AI output"""
        self.messages.append(HumanMessage(content=inputs["input"]))
        self.messages.append(AIMessage(content=outputs["output"]))
    
    def load_memory_variables(self, inputs: dict = None) -> dict:
        """Load conversation history"""
        return {"chat_history": self.messages}

# Memory for conversation
rag_memory = SimpleMemory()

def conversational_rag(question: str):
    """RAG with conversation memory."""
    # Get relevant docs
    docs = retriever.invoke(question)
    context = format_docs(docs)
    
    # Load history
    history = rag_memory.load_memory_variables({})
    
    # Create and run chain
    chain = conversational_rag_prompt | llm | StrOutputParser()
    response = chain.invoke({
        "context": context,
        "question": question,
        "chat_history": history["chat_history"]
    })
    
    # Save to memory
    rag_memory.save_context(
        {"input": question},
        {"output": str(response)}
    )
    
    return response

## 17. Test Conversational RAG

In [17]:
print("🤖 CONVERSATIONAL RAG DEMO\n" + "="*50)

# First question
q1 = "How many vacation days do employees get?"
print(f"\nYou: {q1}")
print(f"Bot: {conversational_rag(q1)}")

# Follow-up question (uses context from previous)
q2 = "And what about senior engineers specifically?"
print(f"\nYou: {q2}")
print(f"Bot: {conversational_rag(q2)}")

# Another follow-up
q3 = "Can those days be carried over?"
print(f"\nYou: {q3}")
print(f"Bot: {conversational_rag(q3)}")

🤖 CONVERSATIONAL RAG DEMO

You: How many vacation days do employees get?
Bot: Employees are entitled to 15, 20, 25, and 30 days of paid vacation. They can carry over up to 10 days per year. Senior staff (Level 4+) receive an additional 5 days annually. Employees must give 2 weeks notice for a vacation longer than 5 days.

You: And what about senior engineers specifically?
Bot: Senior Engineers (Level 4+) participate in an on-call rotation every week, which allows them to work during the night and have a higher probability of being available for critical systems during business hours. In addition, they receive an additional $500 bonus per week for working overtime or attending code reviews. Senior engineers must give at least two weeks' notice for vacation longer than 5 days. For incidents outside of business hours, they are paid an additional compensation of up to $1000 per incident. The process for merging changes also involves a Code Review Process, with peer review, critical systems

## 18. Persisting the Vector Store

In [18]:
# Save to disk
persist_directory = "./chroma_db"

vectorstore_persistent = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="techcorp_docs"
)

print(f"💾 Vector store saved to {persist_directory}")

# Later, load from disk:
# loaded_vectorstore = Chroma(
#     persist_directory=persist_directory,
#     embedding_function=embeddings,
#     collection_name="techcorp_docs"
# )

💾 Vector store saved to ./chroma_db


## 📚 Session 3 Recap

### Key Takeaways:

1. **RAG solves the knowledge problem**
   - LLMs don't know YOUR data
   - RAG = Retrieve relevant docs → Augment prompt → Generate answer

2. **The RAG Pipeline:**
   ```
   Documents → Chunks → Embeddings → Vector Store
                                          ↓
   Question → Retrieve Similar → LLM + Context → Answer
   ```

3. **Key Components:**
   - Document Loaders: Load various formats
   - Text Splitters: Chunk documents
   - Embeddings: Convert text to vectors
   - Vector Stores: Fast similarity search
   - Retrievers: Find relevant documents

4. **Best Practices:**
   - Chunk size: 500-1000 chars typically
   - Overlap: 10-20% of chunk size
   - Top-k: 3-5 documents usually sufficient

---

### 🔜 Next Session: LangSmith - Debugging & Monitoring
"RAG is working, but how do we debug it when things go wrong?"

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                    SESSION 3 COMPLETE! 🎉                                  ║
║                                                                            ║
║  🍽️ LUNCH BREAK - 45 MINUTES 🍽️                                           ║
║                                                                            ║
║  Next: Session 4 - LangSmith (Debugging & Monitoring)                      ║
║  File: 04_langsmith_observability.ipynb                                    ║
║                                                                            ║
║  "RAG is powerful, but how do we see what's happening inside?"             ║
║  "When the chain fails, how do we debug it?"                               ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")